# Dataset Explorer — labeled_v1.parquet

Interactive notebook to explore the LOATO-Bench dataset. Load, inspect, filter, and visualize
the 68.8K samples (40K benign + 29K injection) from 9 public sources.

**No CLI or pipeline knowledge needed** — just run the cells.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)

## 1. Load the Dataset

In [ ]:
df = pd.read_parquet("../data/processed/labeled_v1.parquet")
print(f"Total samples: {len(df):,}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Schema Overview

| Column | Type | Description |
|--------|------|-------------|
| `text` | str | The prompt text |
| `label` | int | 0 = benign, 1 = injection |
| `source` | str | Dataset origin (dolly, oasst, hackaprompt, etc.) |
| `attack_category` | str/None | Attack type slug (C1–C7) for injections, None for benign |
| `language` | str | ISO language code (mostly "en") |
| `is_indirect` | bool | Whether the injection is indirect (via context) |

In [ ]:
df.info()

## 3. Label Distribution

In [ ]:
label_map = {0: "Benign", 1: "Injection"}
label_counts = df["label"].map(label_map).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
label_counts.plot.bar(ax=axes[0], color=["#2ecc71", "#e74c3c"], edgecolor="black")
axes[0].set_title("Label Distribution")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
for i, v in enumerate(label_counts):
    axes[0].text(i, v + 500, f"{v:,}", ha="center", fontweight="bold")

# Pie chart
label_counts.plot.pie(
    ax=axes[1],
    autopct="%1.1f%%",
    colors=["#2ecc71", "#e74c3c"],
    startangle=90,
    textprops={"fontsize": 12},
)
axes[1].set_ylabel("")
axes[1].set_title("Label Balance")

plt.tight_layout()
plt.show()

print(f"Benign:    {(df['label'] == 0).sum():,} ({(df['label'] == 0).mean() * 100:.1f}%)")
print(f"Injection: {(df['label'] == 1).sum():,} ({(df['label'] == 1).mean() * 100:.1f}%)")

## 4. Source Distribution

In [ ]:
source_label = df.groupby(["source", "label"]).size().unstack(fill_value=0)
source_label.columns = ["Benign", "Injection"]
source_label = source_label.sort_values("Benign", ascending=False)

source_label.plot.barh(
    stacked=True, figsize=(10, 5), color=["#2ecc71", "#e74c3c"], edgecolor="black"
)
plt.title("Samples by Source and Label")
plt.xlabel("Count")
plt.legend(title="Label")
plt.tight_layout()
plt.show()

source_label["Total"] = source_label.sum(axis=1)
source_label

## 5. Attack Category Distribution (Injections Only)

In [ ]:
injections = df[df["label"] == 1]
cat_counts = injections["attack_category"].value_counts()

cat_counts.plot.barh(figsize=(10, 4), color="#e74c3c", edgecolor="black")
plt.title("Attack Category Distribution")
plt.xlabel("Count")
for i, (cat, v) in enumerate(cat_counts.items()):
    plt.text(v + 100, i, f"{v:,}", va="center")
plt.tight_layout()
plt.show()

cat_counts

## 6. Sample Texts

Browse example prompts from each category. **Useful for understanding what each attack type looks like.**

In [ ]:
# Benign examples
print("=" * 80)
print("BENIGN SAMPLES (label=0)")
print("=" * 80)
for source in ["dolly", "oasst", "wildchat", "alpaca"]:
    subset = df[(df["label"] == 0) & (df["source"] == source)]
    if len(subset) > 0:
        sample = subset.sample(1, random_state=42).iloc[0]
        print(f"\n[{source}] {sample['text'][:200]}")

In [ ]:
# Injection examples (one per category)
print("=" * 80)
print("INJECTION SAMPLES (label=1)")
print("=" * 80)
for cat in injections["attack_category"].dropna().unique():
    subset = injections[injections["attack_category"] == cat]
    sample = subset.sample(1, random_state=42).iloc[0]
    print(f"\n[{cat}] ({sample['source']})")
    print(f"  {sample['text'][:200]}")

## 7. Text Length Analysis

In [ ]:
df["text_len"] = df["text"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for i, (label_val, label_name, color) in enumerate(
    [(0, "Benign", "#2ecc71"), (1, "Injection", "#e74c3c")]
):
    subset = df[df["label"] == label_val]["text_len"]
    axes[i].hist(subset.clip(upper=2000), bins=50, color=color, edgecolor="black", alpha=0.8)
    axes[i].set_title(f"{label_name} — Text Length Distribution")
    axes[i].set_xlabel("Characters")
    axes[i].set_ylabel("Count")
    axes[i].axvline(
        subset.median(), color="black", linestyle="--", label=f"Median: {subset.median():.0f}"
    )
    axes[i].legend()

plt.tight_layout()
plt.show()

df.groupby("label")["text_len"].describe().round(0)

## 8. Language Distribution

In [ ]:
lang_counts = df["language"].value_counts().head(10)

lang_counts.plot.bar(figsize=(10, 4), color="#3498db", edgecolor="black")
plt.title("Top 10 Languages")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

en_count = (df["language"] == "en").sum()
en_pct = (df["language"] == "en").mean() * 100
non_en = (df["language"] != "en").sum()
non_en_pct = (df["language"] != "en").mean() * 100
print(f"English: {en_count:,} ({en_pct:.1f}%)")
print(f"Non-English: {non_en:,} ({non_en_pct:.1f}%)")

## 9. Filter & Explore

Use this cell to filter the dataset however you want:

In [ ]:
# Examples — uncomment and modify as needed:

# All injection samples from a specific category
# df[df["attack_category"] == "jailbreak_roleplay"].head(10)

# Benign samples from a specific source
# df[(df["label"] == 0) & (df["source"] == "wildchat")].sample(5)

# Non-English samples
# df[df["language"] != "en"].head(10)

# Indirect injection samples
# df[df["is_indirect"] == True].head(10)

# Long texts (>1000 chars)
# df[df["text"].str.len() > 1000].sample(5)

print("Uncomment a filter above and re-run this cell.")

## 10. LOATO Split Summary

How the dataset is split for the LOATO evaluation protocol:

In [ ]:
import json

with open("../data/splits/loato_splits.json") as f:
    splits = json.load(f)

print(f"Experiment: {splits['experiment']}")
print(f"Seed: {splits['seed']}")
print(f"Number of folds: {len(splits['folds'])}")
print()

for i, fold in enumerate(splits["folds"]):
    train_idx = fold["train_indices"]
    test_idx = fold["test_indices"]
    train_labels = df.iloc[train_idx]["label"]
    test_labels = df.iloc[test_idx]["label"]
    cat = fold.get("held_out_category", "?")
    tr_b = int((train_labels == 0).sum())
    tr_i = int((train_labels == 1).sum())
    te_b = int((test_labels == 0).sum())
    te_i = int((test_labels == 1).sum())
    print(f"Fold {i} — held out: {cat}")
    print(f"  Train: {len(train_idx):,} (benign={tr_b:,}, inj={tr_i:,})")
    print(f"  Test:  {len(test_idx):,} (benign={te_b:,}, inj={te_i:,})")
    print()